# EDA on CO2 Emissions Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("co2.csv")
print(df.shape)
df.head()

**Observation:** The dataset has 7,385 rows and 12 columns — each row is one car model/variant.

## 2. Column types and missing values

In [ ]:
df.info()

**Observation:** 5 text columns and 7 numeric columns, with no missing values in any column.

## 3. Summary of numeric columns

In [ ]:
df.describe()

**Observation:** CO2 ranges from 96 to 522 g/km with a mean of about 251; engine size ranges from 0.9 L to 8.4 L.

## 4. Data cleaning
Remove duplicate rows, tidy up the Make/Model text, and rename the columns to short names.

In [ ]:
print("Missing values:\n", df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())

df["Model"] = df["Model"].str.strip().str.upper()
df["Make"] = df["Make"].str.strip().str.upper()
df = df.drop_duplicates().reset_index(drop=True)
df.columns = ["make", "model", "vclass", "engine", "cylinders", "trans","fuel", "city", "hwy", "comb", "comb_mpg", "co2"]

**Observation:** There are no missing values, and 1,103 duplicate rows were removed (the same cars repeat across model years), leaving 5,990 rows.

## 5. Distribution of CO2 emissions

In [ ]:
plt.hist(df["co2"],bins = 40, edgecolor="black")
plt.title("CO2 distribution")
plt.xlabel("CO2 (g/km)")
plt.ylabel("Count")
plt.show()

**Observation:** Most cars emit 208–290 g/km (median 246). The distribution is slightly right-skewed (skew ≈ 0.56) because of a few very high emitters such as supercars and large vans.

## 6. Counts of categorical features

In [ ]:
for col in ["fuel", "vclass", "trans", "cylinders"]:
    df[col].value_counts().plot(kind="bar", title=col)
    plt.tight_layout()
    plt.show()

df["make"].value_counts().head(15).plot(kind="barh", title="Top 15 makes")
plt.show()

**Observation:** Regular (X) and premium (Z) gasoline make up over 90% of cars, while natural gas (N) has only 1 car. 4-, 6- and 8-cylinder engines dominate, and Ford, Chevrolet and BMW have the most models.

## 7. Numeric features vs CO2

In [ ]:
for col in ["engine", "cylinders", "comb", "city", "hwy"]:
    for fuel, grp in df.groupby("fuel"):
        plt.scatter(grp[col], grp["co2"], label=fuel, alpha=0.5, s=12)
    plt.xlabel(col) 
    plt.ylabel("CO2 (g/km)")
    plt.legend()
    plt.title(f"{col} vs CO2") 
    plt.show()

**Observation:** CO2 rises almost linearly with fuel consumption and also increases with engine size and cylinders. Each fuel type forms its own line: ethanol (E) cars burn more fuel but emit less CO2 per litre, while diesel (D) emits the most CO2 per litre.

## 8. Correlation

In [ ]:
corr = df.select_dtypes("number").corr()

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr)))
ax.set_yticklabels(corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im)
plt.title("Correlation matrix")
plt.tight_layout()
plt.show()

pd.plotting.scatter_matrix(df[["engine", "cylinders", "comb", "co2"]], figsize=(9, 9), alpha=0.4)
plt.show()

**Observation:** City, highway and combined fuel consumption are the features most strongly correlated with CO2 (≈ 0.88–0.92), followed by engine size (0.85) and cylinders (0.84). City, highway, combined and mpg are almost duplicates of each other (city vs combined = 0.99), so only one of them is needed for modelling.

## 9. Conclusions

- **Data:** 7,385 rows, 12 columns, no missing values. 1,103 duplicate rows were removed, leaving 5,990 cars.
- **CO2:** Ranges from 96 to 522 g/km (median 246). It is slightly right-skewed because of a few very high emitters.
- **Categories:** Regular and premium gasoline make up over 90% of cars. Most cars have 4, 6 or 8 cylinders.
- **Key drivers:** Fuel consumption is most strongly linked to CO2 (≈ 0.92), followed by engine size (0.85) and cylinders (0.84).
- **Fuel type:** Diesel releases the most CO2 per litre and ethanol the least.
- **Redundancy:** City, highway and combined consumption, and mpg, are near-duplicates, so only one is needed for modelling.

**Summary:** CO2 depends mainly on how much fuel a car burns, adjusted by fuel type.
